# Intrusion detection and prevention systems (IDS/IPS)  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 3.5.a: IDS signature matcher

**Skill:** Detect Attacks · **Maps to:** 3.5.A · **Language:** Python 3 (pytest)

## Problem
You are implementing the rule-matching core of a simple IDS. The sensor receives a stream of
packet-log entries. Each entry is a dict with the fields shown below. You are given a list of
signature rules; each rule has a destination port and a keyword that must appear (case-insensitive)
in the payload. A log entry matches a rule when BOTH the port AND the keyword condition are met.

Return the sorted list of `id` values for every log entry that matches at least one rule.

## Function contract
Implement `solve(logs: list[dict], rules: list[dict]) -> list[str]` in the code editor
(see the Reference section below).

## Input
- `logs`: list of dicts, each with keys:
  - `"id"` (str), unique entry identifier, e.g. `"pkt-001"`
  - `"dst_port"` (int), destination port of the packet
  - `"payload"` (str), packet payload (may be empty)
- `rules`: list of dicts, each with keys:
  - `"port"` (int), destination port to match
  - `"keyword"` (str), substring to find (case-insensitive) in the payload

## Output
A sorted `list[str]` of `id` values for entries that match at least one rule.
Empty list if no entries match.

## Examples
```
logs = [
    {"id": "pkt-001", "dst_port": 80,  "payload": "GET /admin/../etc/passwd HTTP/1.1"},
    {"id": "pkt-002", "dst_port": 443, "payload": "TLS handshake"},
    {"id": "pkt-003", "dst_port": 80,  "payload": "GET /index.html HTTP/1.1"},
    {"id": "pkt-004", "dst_port": 22,  "payload": "SSH-2.0-OpenSSH_8.9"},
]
rules = [
    {"port": 80, "keyword": "etc/passwd"},
    {"port": 22, "keyword": "libssh"},
]
solve(logs, rules)  ->  ["pkt-001"]

logs = []
solve(logs, rules)  ->  []
```

## Constraints
- Deterministic, offline, standard library only.
- Keyword matching is case-insensitive substring match.
- A log entry that matches multiple rules is included only once in the output.
- Output is sorted ascending by `id`.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(logs: list[dict], rules: list[dict]) -> list[str]
```

### Input (`logs`, `rules`)
- `logs`: list of dicts with keys:
  - `"id"` (str), unique packet/log entry identifier
  - `"dst_port"` (int), destination port
  - `"payload"` (str), packet payload text
- `rules`: list of dicts with keys:
  - `"port"` (int), destination port to match
  - `"keyword"` (str), substring to search for in payload

### Output (`result`)
A sorted `list[str]` of `id` values for log entries that match at least one rule.
Empty list if no entries match.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Keyword matching is case-insensitive (`keyword.lower() in payload.lower()`).
- An entry matching multiple rules appears only once (de-duplicated).
- Output is sorted ascending by `id` string value.
- A log entry matches a rule when its `dst_port == rule["port"]` AND `rule["keyword"].lower()` is a substring of `entry["payload"].lower()`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 3.5.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import List, Dict, Any


def solve(logs: List[Dict[str, Any]], rules: List[Dict[str, Any]]) -> List[str]:
    """Return sorted list of log entry IDs that match at least one IDS signature rule.

    A log entry matches a rule when its dst_port equals the rule's port AND
    the rule's keyword appears (case-insensitive) in the entry's payload.
    """
    matched = set()
    for entry in logs:
        port = entry["dst_port"]
        payload_lower = entry["payload"].lower()
        for rule in rules:
            if rule["port"] == port and rule["keyword"].lower() in payload_lower:
                matched.add(entry["id"])
                break  # one match is enough; avoid duplicates
    return sorted(matched)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- shared fixtures ---------------------------------------------------------
RULES_BASIC = [
    {"port": 80,  "keyword": "etc/passwd"},
    {"port": 22,  "keyword": "libssh"},
    {"port": 3306, "keyword": "select "},
]

LOGS_BASIC = [
    {"id": "pkt-001", "dst_port": 80,   "payload": "GET /admin/../etc/passwd HTTP/1.1"},
    {"id": "pkt-002", "dst_port": 443,  "payload": "TLS handshake data"},
    {"id": "pkt-003", "dst_port": 80,   "payload": "GET /index.html HTTP/1.1"},
    {"id": "pkt-004", "dst_port": 22,   "payload": "SSH-2.0-OpenSSH_8.9"},
    {"id": "pkt-005", "dst_port": 3306, "payload": "SELECT * FROM users"},
    {"id": "pkt-006", "dst_port": 22,   "payload": "SSH-2.0-libssh_0.9.6"},
]


# ---- visible cases -----------------------------------------------------------
def test_path_traversal_match():
    result = impl.solve(LOGS_BASIC, RULES_BASIC)
    assert "pkt-001" in result, "pkt-001 matches port=80 + keyword='etc/passwd'"


def test_benign_port_no_match():
    result = impl.solve(LOGS_BASIC, RULES_BASIC)
    assert "pkt-002" not in result, "pkt-002 is port 443 — no rule for that port"
    assert "pkt-003" not in result, "pkt-003 port 80 but payload has no matching keyword"


def test_sql_case_insensitive():
    # "SELECT" (uppercase) must match keyword "select " (lowercase)
    result = impl.solve(LOGS_BASIC, RULES_BASIC)
    assert "pkt-005" in result, "SQL SELECT should match keyword 'select ' case-insensitively"


def test_libssh_match():
    result = impl.solve(LOGS_BASIC, RULES_BASIC)
    assert "pkt-006" in result, "pkt-006 port=22 payload contains 'libssh'"
    assert "pkt-004" not in result, "pkt-004 port=22 but 'OpenSSH' does not contain 'libssh'"


def test_result_sorted():
    result = impl.solve(LOGS_BASIC, RULES_BASIC)
    assert result == sorted(result), "Output must be sorted ascending"


# ---- hidden / edge cases -----------------------------------------------------
def test_empty_logs():
    assert impl.solve([], RULES_BASIC) == []


def test_empty_rules():
    assert impl.solve(LOGS_BASIC, []) == []


def test_both_empty():
    assert impl.solve([], []) == []


def test_no_payload_match():
    logs = [{"id": "x-001", "dst_port": 80, "payload": ""}]
    rules = [{"port": 80, "keyword": "admin"}]
    assert impl.solve(logs, rules) == []


def test_multiple_rules_same_entry_deduplicated():
    # pkt-A matches two rules; should appear only once
    logs = [{"id": "pkt-A", "dst_port": 80, "payload": "etc/passwd and script alert"}]
    rules = [
        {"port": 80, "keyword": "etc/passwd"},
        {"port": 80, "keyword": "script"},
    ]
    result = impl.solve(logs, rules)
    assert result.count("pkt-A") == 1, "Entry matching multiple rules must appear only once"


def test_port_must_match():
    # keyword present but wrong port — should not match
    logs = [{"id": "z-001", "dst_port": 443, "payload": "etc/passwd in body"}]
    rules = [{"port": 80, "keyword": "etc/passwd"}]
    assert impl.solve(logs, rules) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
